# Error Analysis

## Imports & Load data

In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score
from pathlib import Path

df_train = pd.read_csv('../data/work/train.csv')
df_test  = pd.read_csv('../data/work/test.csv')
X_train = df_train.drop(columns=['total'])
y_train = df_train['total'].values
X_test  = df_test.drop(columns=['total'])
y_test  = df_test['total'].values
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## Build evaluation Dataframe

In [3]:
cat_cols = ["address", "district", "type"]
num_cols = ["bedrooms", "garage"]
area_col = ["area"]  

for df in (X_train, X_test):
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip().str.lower()
        
y_train = pd.to_numeric(y_train, errors="coerce")

p99_area = np.nanpercentile(X_train["area"].to_numpy(), 99)

random_forest = RandomForestRegressor(random_state=42, n_jobs=-1)

area_tree = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("clip99", FunctionTransformer(
        lambda X: np.clip(X, None, p99_area),
        feature_names_out="one-to-one"
    )),
])

num_tree = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
])

ct_tree = ColumnTransformer(
    transformers=[
        ("num_other", num_tree, num_cols),
        ("area",      area_tree, area_col),  
        ("cat",       cat_pipe,  cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

pipe_forest_tunned = Pipeline(steps=[
    ("preprocess", ct_tree),       
    ("model", random_forest)
])


param_dist = {
    "model__n_estimators": [300, 600, 900],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_leaf": [1, 2, 4],
    "model__min_samples_split": [2, 5, 10],
    "model__max_features": ["sqrt", 0.5],
    "model__bootstrap": [True],
}

random_search_forest = RandomizedSearchCV(
    estimator=pipe_forest_tunned,                   
    param_distributions=param_dist,                
    n_iter=20,
    scoring="neg_mean_absolute_error",             
    cv=5,
    random_state=42,
    n_jobs=-1,
    error_score=np.nan                              
)

random_search_forest.fit(X_train, y_train)

best_mae_forest   = -random_search_forest.best_score_
best_params_forest = random_search_forest.best_params_
best_forest_pipe = random_search_forest.best_estimator_

y_pred = best_forest_pipe.predict(X_test)
y_true_s = pd.Series(y_test, index=X_test.index, name="y_true")
y_pred_s = pd.Series(y_pred,   index=X_test.index, name="y_pred")

eval_df = pd.concat([X_test.reset_index(drop=True),
                     y_true_s.reset_index(drop=True),
                     y_pred_s.reset_index(drop=True)], axis=1)

eval_df["error"] = eval_df["y_true"] - eval_df["y_pred"]
eval_df["abs_error"] = eval_df["error"].abs()

eval_df.head(3)

,area,bedrooms,garage,address,district,type,y_true,y_pred,error,abs_error
0,125,2,2,rua floro de oliveira,jardim adriana,casa em condomínio,1878,4538.096215,-2660.096215,2660.096215
1,70,2,1,praça benedito calixto,pinheiros,apartamento,5387,8149.294996,-2762.294996,2762.294996
2,67,2,1,rua alvorada,vila olímpia,apartamento,5139,3707.261795,1431.738205,1431.738205


## Global metrics (test split)

In [4]:
mae   = mean_absolute_error(eval_df["y_true"], eval_df["y_pred"])
medae = median_absolute_error(eval_df["y_true"], eval_df["y_pred"])
r2    = r2_score(eval_df["y_true"], eval_df["y_pred"])
p90   = eval_df["abs_error"].quantile(0.90)
p95   = eval_df["abs_error"].quantile(0.95)
bias  = eval_df["error"].mean()  

print(f"Test MAE:   {mae:.2f}")
print(f"Test MedAE: {medae:.2f}")
print(f"Test R²:    {r2:.4f}")
print(f"P90 |error|:{p90:.2f}")
print(f"P95 |error|:{p95:.2f}")
print(f"Bias (mean error y_true - y_pred): {bias:.2f}")


Test MAE:   1233.68
Test MedAE: 752.97
Test R²:    0.6853
P90 |error|:2785.93
P95 |error|:4116.68
Bias (mean error y_true - y_pred): -58.12


## Segments summaries (district, address)
**Goal:** find groups with systematically larger errors.

**Rule of thumb:** only show groups with enough data (e.g., `n ≥ 50`).

In [5]:
def segment_summary_simple(df, by, min_n=50):
    g = (df.groupby(by)
           .agg(n=('abs_error','size'),
                mae=('abs_error','mean'),
                bias=('error','mean'))
           .reset_index())

    g = g[g['n'] >= min_n]

    return g.sort_values('mae', ascending=False)

In [6]:
reports = {}

if "district" in eval_df.columns:
    district_report = segment_summary_simple(eval_df, by="district", min_n=50)
    district_report.to_csv(REPORTS_DIR / "b52_top_district_mae.csv", index=False)
    reports["top_district"] = district_report.head(10)
    print("Saved reports/b52_top_district_mae.csv")

if "address" in eval_df.columns:
    address_report = segment_summary_simple(eval_df, by="address", min_n=50)
    address_report.to_csv(REPORTS_DIR / "b52_top_address_mae.csv", index=False)
    reports["top_address"] = address_report.head(10)
    print("Saved reports/b52_top_state_mae.csv")

reports.get("top_district", pd.DataFrame()).head(10), reports.get("top_address", pd.DataFrame()).head(10)

Saved reports/b52_top_district_mae.csv
Saved reports/b52_top_state_mae.csv


(      district   n          mae        bias
 15  bela vista  58  1101.948241  312.279991,
 Empty DataFrame
 Columns: [address, n, mae, bias]
 Index: [])

## Price quintiles (bins by true price)

In [7]:
eval_df["price_quintile"] = pd.qcut(eval_df["y_true"], q=5, duplicates="drop")

price_quintile_report = (eval_df
    .groupby("price_quintile")
    .agg(n=('abs_error','size'),
         mae=('abs_error','mean'),
         bias=('error','mean'))
    .reset_index())

# Keep quantile order (categorical codes)
if hasattr(price_quintile_report["price_quintile"], "cat"):
    price_quintile_report = price_quintile_report.sort_values(
        by="price_quintile",
        key=lambda s: s.cat.codes
    )

price_quintile_report.to_csv(REPORTS_DIR / "b52_price_quintiles.csv", index=False)
price_quintile_report.head()

C:\Users\henrique.nascimento\AppData\Local\Temp\ipykernel_58856\2340229226.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("price_quintile")


,price_quintile,n,mae,bias
0,"(520.999, 1796.4]",467,694.390969,-651.712528
1,"(1796.4, 2597.2]",466,709.794499,-585.541851
2,"(2597.2, 3614.4]",466,813.586883,-375.942644
3,"(3614.4, 5431.2]",466,1190.001178,-88.510753
4,"(5431.2, 28700.0]",467,2758.491033,1409.241284


## Area quitiles 

In [9]:
area = pd.to_numeric(eval_df["area"], errors="coerce")
qbins = np.quantile(area.dropna(), [0, .2, .4, .6, .8, 1.0]) if area.notna().any() else np.array([])
qbins = np.unique(qbins)

if qbins.size >= 3:
    bins = qbins
else:
    bins = np.array([0, 400, 800, 1200, 1600, 2500, np.inf])

eval_df["area_bin"] = pd.cut(area, bins=bins, include_lowest=True)

area_report = (eval_df.groupby("area_bin")
    .agg(n=('abs_error','size'),
         mae=('abs_error','mean'),
         bias=('error','mean'))
    .reset_index())

area_report.to_csv(REPORTS_DIR / "b52_area_bins.csv", index=False)
area_report.head()


C:\Users\henrique.nascimento\AppData\Local\Temp\ipykernel_58856\2412660457.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  area_report = (eval_df.groupby("area_bin")


,area_bin,n,mae,bias
0,"(0.999, 39.0]",482,783.762909,33.168669
1,"(39.0, 52.0]",474,700.655222,-151.926106
2,"(52.0, 70.0]",447,899.362174,-44.274764
3,"(70.0, 112.8]",462,1352.288032,-86.560022
4,"(112.8, 550.0]",467,2441.704300,-42.236898


## Calibration by prediction deciles

In [11]:
eval_df["pred_decile"] = pd.qcut(eval_df["y_pred"], q=10, duplicates="drop")

calib = (eval_df
    .groupby("pred_decile")
    .agg(n=('y_true','size'),
         y_true_mean=('y_true','mean'),
         y_pred_mean=('y_pred','mean'),
         mae=('abs_error','mean'),
         bias=('error','mean'))
    .reset_index())

# Keep decile order
if hasattr(calib["pred_decile"], "cat"):
    calib = calib.sort_values(by="pred_decile", key=lambda s: s.cat.codes)

calib.to_csv(REPORTS_DIR / "b53_calibration_by_pred_decile.csv", index=False)
calib.head()

C:\Users\henrique.nascimento\AppData\Local\Temp\ipykernel_58856\2938055016.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("pred_decile")


,pred_decile,n,y_true_mean,y_pred_mean,mae,bias
0,"(962.839, 1865.652]",235,1428.697872,1445.560632,446.308429,-16.862760
1,"(1865.652, 2210.347]",233,2041.030043,2061.162010,645.707122,-20.131967
2,"(2210.347, 2445.264]",232,2301.883621,2316.060721,744.850767,-14.177100
3,"(2445.264, 2818.921]",233,2563.017167,2618.758799,682.757821,-55.741631
4,"(2818.921, 3277.5]",233,3088.493562,3055.586923,808.852048,32.906639


## Outliers: Top 20 absolute errors

In [13]:
top20 = eval_df.nlargest(20, "abs_error").copy()

area = pd.to_numeric(top20["area"], errors="coerce")
top20["flag_sqft_unusually_large"] = area > area.quantile(0.99)
top20["flag_sqft_nonpositive"]     = (area <= 0) | (area.isna())

beds  = pd.to_numeric(top20["bedrooms"], errors="coerce")
top20["flag_bed_implausible"]  = (beds < 0) | (beds > 10)


type_cols = [c for c in top20.columns if c.startswith("type_")]
top20[type_cols] = top20[type_cols].fillna(False)
top20["type"] = top20[type_cols].apply(lambda r: "|".join([c for c, v in r.items() if bool(v)]), axis=1)

cols_show = [
    "y_true","y_pred","error","abs_error",
    "address","district","area","bedrooms","garage","type"
]
cols_show = [c for c in cols_show if c in top20.columns]

out_path = REPORTS_DIR / "b54_top20_abs_errors.csv"
top20[cols_show + type_cols].to_csv(out_path, index=False)
print("Saved", out_path)
top20[cols_show + type_cols].head()

Saved ..\reports\b54_top20_abs_errors.csv


,y_true,y_pred,error,abs_error,address,district,area,bedrooms,garage,type
1801,26710,2293.990541,24416.009459,24416.009459,avenida chibarás,planalto paulista,24,1,0,
1355,16680,4529.412595,12150.587405,12150.587405,rua senador césar lacerda vergueiro,sumarezinho,70,1,1,
1463,28700,17345.316927,11354.683073,11354.683073,alameda dos tupiniquins,planalto paulista,454,4,5,
647,13200,3122.777402,10077.222598,10077.222598,"r. nossa sra. operária, 286 - vila guilherme, ...",vila guilherme,125,3,0,
2314,13260,3598.952175,9661.047825,9661.047825,rua tonelero,vila ipojuca,150,2,1,
